# Hubble–Lemaître and HR diagrams

The SDSS keeps its catalogs in a relational database: tables of rows and columns that are queried with SQL. **CasJobs** is the web service that runs such queries. SciScript's `CasJobs.executeQuery` sends the text of a query and returns the result as a pandas DataFrame. The *context* names the database the query runs in; these notebooks use the DR17 release (`settings.CASJOBS_CONTEXT`). The queries here need no SciServer account.

This notebook draws two classic diagrams of astronomy:

1. **Hubble–Lemaître diagram:** the brightness of galaxies of the same kind against their redshift, which shows that distance grows in proportion to redshift.
2. **Hertzsprung–Russell (HR) diagram:** the luminosity of nearby stars against their color, beside the temperature and surface gravity measured from their spectra.

Database ideas in this notebook:

- `SELECT ... FROM ... WHERE ...`: which columns, from which table, for which rows;
- joining two tables on a shared key;
- testing flag bits;
- computing new columns inside the query.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy import constants
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.ticker import (
    FormatStrFormatter,
    LogLocator,
    NullFormatter,
    StrMethodFormatter,
)
from SciServer import CasJobs

import settings

# The queries in this notebook need no SciServer login. Silence the
# warning SciScript gives about the missing login token.
warnings.filterwarnings("ignore", message="In Authentication.getToken")

## 1. Hubble–Lemaître diagram

The Hubble–Lemaître law says that galaxies recede with a speed proportional to their distance, $v = H_{0} d$. The speed is measured from the redshift $z$, as $v \approx c z$ for nearby galaxies. Testing the law needs distances measured some other way: from **standard candles**, objects whose luminosity is known. The flux of a standard candle falls as the inverse square of its distance. If distance is proportional to redshift, the flux falls as $1 / z^{2}$, and the apparent magnitude rises as $5 \log_{10} z$ plus a constant. Against $\log z$, standard candles of one kind then lie on a straight line of slope $5$.

### Standard candles

The stars of an elliptical galaxy move in random directions. The spread of their speeds along the line of sight, the **velocity dispersion** $\sigma$, broadens the absorption lines in the galaxy's spectrum, so it is measured without knowing the distance. Ellipticals with the same velocity dispersion have about the same luminosity (the Faber–Jackson relation), with a spread of about $0.5$ magnitudes. A narrow range of $\sigma$ therefore selects a family of standard candles, and several ranges give several families.

### The query

Two tables hold most of what the SDSS measured:

- `SpecObj` has one row per spectrum: the redshift `z`, the velocity dispersion `velDisp`, quality flags and the targeting bits that say why the spectrum was taken.
- `PhotoObj` has one row per object detected in the imaging: magnitudes, sizes, shapes and flags.

The column `bestObjID` of `SpecObj` holds the `objID` of the imaged object each spectrum belongs to. `JOIN PhotoObj AS p ON p.objID = s.bestObjID` pairs every spectrum with its object; `s` and `p` are short names (aliases) for the two tables. A third table, `Photoz`, holds photometric redshift estimates and, with them, the k-corrections used below. A few galaxies have no `Photoz` row; the plain `JOIN` drops them, which changes nothing here.

The `WHERE` conditions select:

- the **Main Galaxy Sample** of SDSS-I/II: spectra whose `legacy_target1` column has the `GALAXY` bit set. `dbo.fPrimTarget('GALAXY')` returns that bit, and `&` tests it;
- spectra classified as galaxies, with a redshift fit free of warnings (`zWarning = 0`);
- galaxies whose light profile is mostly that of an elliptical: `fracDeV_r` is the weight of the de Vaucouleurs profile, typical of ellipticals, in the best fit combining it with the exponential profile of a disk;
- the redshift range and the velocity dispersion range of the parameter set.

The query does not require `p.clean = 1`, the pipeline's mark of trouble-free photometry, which the other notebooks use. Among these ellipticals the unclean ones mostly touch the edge of an imaging frame, contain saturated pixels or overlap a neighbor. Nearer than $z \approx 0.04$ they are typically about $0.3$ magnitudes brighter than the clean galaxies of the same family, and they make up about a quarter of the galaxies below $z = 0.03$ but only a tenth at $z = 0.05$. Removing them would take more bright galaxies from near than from far and bend the very trend this diagram measures: with the cut, the slope of the fastest family drops from $4.9$ to $4.7$.

The `SELECT` list computes two columns on the server:

- `sigma`, the velocity dispersion corrected for the size of the fiber. The SDSS spectrograph sees each galaxy through a fiber $3''$ across. For a nearby galaxy that is the bright center, for a distant one a larger part of the galaxy, where the stars move more slowly. The correction scales every dispersion to the value within one eighth of the galaxy's effective radius, `deVRad_r`. It uses only angles on the sky, so no distance enters it. `NULLIF(p.deVRad_r, 0)` turns a zero radius into `NULL`, so the division gives `NULL` instead of an error.
- `rest_r`, the $r$ magnitude corrected for Galactic dust (`dered_r`) and for redshift. Redshift moves a different part of each galaxy's spectrum into the $r$ filter; the **k-correction** `kcorrR` from `Photoz` converts the magnitude to the value it would have at $z = 0$. `Photoz` computes it at the photometric redshift, which differs from the spectroscopic one by about $0.01$ for these galaxies.

SQL does not let `WHERE` refer to a name defined in the `SELECT` list, so the expression for `sigma` appears in both places. Python builds it once.

In [ ]:
hubble_limits = settings.parameter_set("hubble_lemaitre_diagram")
sigma_edges = hubble_limits["sigma_edges"]
sigma = (
    f"s.velDisp * POWER({settings.SDSS_FIBER_RADIUS} "
    f"/ ({settings.APERTURE_CORRECTION_FRACTION} * NULLIF(p.deVRad_r, 0)), "
    f"{settings.APERTURE_CORRECTION_EXPONENT})"
)
hubble_query = f"""
SELECT
    s.z,
    {sigma} AS sigma,
    p.dered_r - pz.kcorrR AS rest_r
FROM SpecObj AS s
    JOIN PhotoObj AS p ON p.objID = s.bestObjID
    JOIN Photoz AS pz ON pz.objID = s.bestObjID
WHERE
    (s.legacy_target1 & dbo.fPrimTarget('GALAXY')) > 0
    AND s.class = 'GALAXY'
    AND s.zWarning = 0
    AND p.fracDeV_r > {hubble_limits["min_frac_dev"]}
    AND s.z BETWEEN {hubble_limits["z_min"]} AND {hubble_limits["z_max"]}
    AND {sigma} >= {sigma_edges[0]}
    AND {sigma} < {sigma_edges[-1]}
"""
print(hubble_query)

In [ ]:
%%time
galaxies = CasJobs.executeQuery(
    sql=hubble_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
print(f"{len(galaxies):,} galaxies")
galaxies.head()

### One line per family

pandas sorts the galaxies into families by their corrected velocity dispersion. Within the fit range of redshift, a least-squares straight line of magnitude against $\log_{10} z$ gives each family's slope, with its statistical error. The fit range leaves out the nearest galaxies, which are few and fall below the trend (see below), and stops where the faintest members of the slowest family start to drop below the survey's limit.

The $\pm$ of each slope is its statistical error alone; moving either end of the fit range by $0.01$–$0.02$ changes the slopes by up to $0.2$. The `offset` places a line of slope exactly $5$ through the median of the same galaxies; the diagram draws it for comparison.

In [ ]:
family_labels = [
    f"{low}–{high}" for low, high in zip(sigma_edges, sigma_edges[1:])
]
galaxies["family"] = pd.cut(
    galaxies["sigma"], sigma_edges, right=False, labels=family_labels
)
in_fit_range = galaxies["z"].between(
    hubble_limits["fit_z_min"], hubble_limits["fit_z_max"]
)

fits = []
for label, members in galaxies[in_fit_range].groupby("family", observed=True):
    log_z = np.log10(members["z"])
    (slope, _), covariance = np.polyfit(log_z, members["rest_r"], 1, cov=True)
    fits.append(
        {
            "family": label,
            "galaxies": len(members),
            "slope": slope,
            "slope_error": np.sqrt(covariance[0, 0]),
            "offset": np.median(members["rest_r"] - 5 * log_z),
        }
    )
fits = pd.DataFrame(fits).set_index("family")
fits.round(2)

### The diagram

In the upper panel each dot is one galaxy, colored by its family; magnitudes grow toward fainter galaxies, so fainter is up. The circles are the median magnitude of each family in narrow steps of $\log z$, with the statistical error of the median: filled inside the fit range, open outside it. The lines have slope $5$. The lower panel subtracts each family's line from its medians. Where a family follows the Hubble–Lemaître law its medians lie on the zero line; $\Delta m_r > 0$ means fainter than the law predicts. The top axis converts redshift to recession velocity, $cz$.

In [ ]:
step = hubble_limits["log_z_step"]
log_z_edges = np.arange(
    np.log10(hubble_limits["z_min"]),
    np.log10(hubble_limits["z_max"]) + step,
    step,
)
z_centers = 10 ** ((log_z_edges[:-1] + log_z_edges[1:]) / 2)
galaxies["z_center"] = pd.cut(
    np.log10(galaxies["z"]), log_z_edges, labels=z_centers
).astype(float)
steps = galaxies.groupby(["family", "z_center"], observed=True)["rest_r"]
# Statistical error of a median: sqrt(pi / 2) times that of a mean,
# with the spread taken from the 16th and 84th percentiles.
spread = (steps.quantile(0.84) - steps.quantile(0.16)) / 2
medians = pd.DataFrame(
    {
        "rest_r": steps.median(),
        "error": np.sqrt(np.pi / 2) * spread / np.sqrt(steps.size()),
        "galaxies": steps.size(),
    }
).reset_index()
medians = medians[
    medians["galaxies"] >= hubble_limits["min_galaxies_per_step"]
]

fit_z_min = hubble_limits["fit_z_min"]
fit_z_max = hubble_limits["fit_z_max"]
z_range = np.array([hubble_limits["z_min"], hubble_limits["z_max"]])
family_colors = dict(zip(family_labels, settings.OKABE_ITO[1:]))
range_labels = {
    label: f"${low}$–${high}$"
    for label, low, high in zip(family_labels, sigma_edges, sigma_edges[1:])
}
shuffled = galaxies.sample(frac=1, random_state=0)
speed_of_light = constants.c.to("km/s").value

with plt.style.context(settings.LIGHT_STYLE):
    fig, (ax, residual_ax) = plt.subplots(
        2,
        1,
        figsize=(0.6 * settings.FIGURE_WIDTH, 0.6 * settings.FIGURE_WIDTH),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )
    fig.get_layout_engine().set(h_pad=0, hspace=0)
    ax.scatter(
        shuffled["z"],
        shuffled["rest_r"],
        c=shuffled["family"].map(family_colors).astype(str),
        s=1.5,
        alpha=0.35,
        linewidths=0,
        rasterized=True,
    )
    legend_handles = []
    for label, fit in fits.iterrows():
        color = family_colors[label]
        family_medians = medians[medians["family"] == label]
        z_center = family_medians["z_center"]
        law = fit.offset + 5 * np.log10(z_center)
        inside = z_center.between(fit_z_min, fit_z_max)
        marker_styles = [
            (inside, {"markerfacecolor": color, "markeredgecolor": "black"}),
            (~inside, {"markerfacecolor": "white", "markeredgecolor": color}),
        ]
        panels = [
            (ax, family_medians["rest_r"]),
            (residual_ax, family_medians["rest_r"] - law),
        ]
        for panel, values in panels:
            for selected, marker_style in marker_styles:
                panel.errorbar(
                    z_center[selected],
                    values[selected],
                    yerr=family_medians["error"][selected],
                    fmt="o",
                    ecolor=color,
                    zorder=3,
                    **marker_style,
                )
        ax.plot(
            z_range,
            fit.offset + 5 * np.log10(z_range),
            color=color,
            zorder=2,
        )
        legend_label = (
            f"{range_labels[label]}:  slope "
            rf"${fit.slope:.2f} \pm {fit.slope_error:.2f}$"
        )
        legend_handles.append(
            Line2D(
                [],
                [],
                color=color,
                marker="o",
                markeredgecolor="black",
                label=legend_label,
            )
        )
    lowest = (medians["rest_r"] - medians["error"]).min()
    ax.set_xscale("log")
    ax.set_xlim(*z_range)
    ax.set_ylim(lowest - 0.2, galaxies["rest_r"].quantile(0.998))
    ax.tick_params(axis="x", which="both", top=False)
    ax.set_ylabel("k-corrected apparent magnitude $m_r$")
    ax.legend(
        handles=legend_handles,
        title=r"velocity dispersion $\sigma$ ($\mathrm{km\,s^{-1}}$)",
        loc="upper left",
    )
    velocity_axis = ax.secondary_xaxis(
        "top",
        functions=(
            lambda z: z * speed_of_light,
            lambda v: v / speed_of_light,
        ),
    )
    velocity_axis.xaxis.set_major_locator(LogLocator(subs=(1, 2, 3, 5)))
    velocity_axis.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    velocity_axis.xaxis.set_minor_formatter(NullFormatter())
    velocity_axis.set_xlabel(
        r"recession velocity $cz$ ($\mathrm{km\,s^{-1}}$)"
    )
    residual_ax.axhline(
        0,
        color=plt.rcParams["axes.edgecolor"],
        linewidth=plt.rcParams["axes.linewidth"],
    )
    residual_ax.set_ylim(-0.75, 0.75)
    residual_ax.xaxis.set_major_locator(LogLocator(subs=(1, 2, 3, 5)))
    residual_ax.xaxis.set_major_formatter(FormatStrFormatter("%g"))
    residual_ax.xaxis.set_minor_formatter(NullFormatter())
    residual_ax.set_xlabel("redshift $z$")
    residual_ax.set_ylabel(r"residual $\Delta m_r$")
    fig.align_ylabels([ax, residual_ax])
    plt.show()

What to look for:

- **Parallel bands.** Each family forms its own band, and the families with faster-moving stars lie lower: they are more luminous. This is the Faber–Jackson relation at work.
- **Slope $5$.** Within the fit range the fitted slopes lie within $0.1$ of $5$, and the filled circles of the lower panel stay within $0.1$ magnitudes of zero: the families fade by $5$ magnitudes for every factor of ten in redshift. Their flux falls as $1 / z^{2}$, so their distance grows in proportion to $z$: the Hubble–Lemaître law. The line of slope $5$ is the law itself; the full expansion history of the Planck 2018 cosmology would steepen it only slightly, to about $5.17$ over this range.
- **The survey limit.** The Main Galaxy Sample contains only galaxies brighter than $r = 17.77$ (Petrosian magnitude). Beyond $z \approx 0.08$ the faintest members of each family start to fall below it, so the medians turn brighter than the lines and drop below zero in the lower panel, the slowest (faintest) family first.
- **The nearest galaxies.** Below $z \approx 0.03$ the medians lie above zero, fainter than the lines, with large errors. Few galaxies are this near, and the fit range leaves them out.

The diagram shows that distance is proportional to redshift, but not the constant of proportionality, $H_{0}$. That needs the true luminosity of one family, calibrated on galaxies whose distances are known by other means.

## 2. Hertzsprung–Russell diagram

The HR diagram plots the luminosity of stars against their temperature. Most stars lie on the **main sequence**, where they burn hydrogen in their cores; giants and white dwarfs lie above and below it. Luminosity needs a distance. The Gaia satellite measures the **parallax** of stars, the small yearly shift of their position as the Earth orbits the Sun; the distance in parsecs is $1000$ divided by the parallax in milliarcseconds. The absolute magnitude, the magnitude a star would have at $10$ parsecs, is then

$$M_G = G + 5 \log_{10}(\varpi / 100),$$

with $G$ the Gaia magnitude and $\varpi$ the parallax in milliarcseconds. The color $\mathrm{BP} - \mathrm{RP}$, the difference between Gaia's blue and red magnitudes, stands in for the temperature.

The stars come from **APOGEE**, the SDSS infrared spectrograph that surveyed stars of the Milky Way. Two of its tables are joined on `apstar_id`:

- `apogeeStar` has one row per star and telescope, with the star's Gaia measurements copied in (columns starting with `gaiaedr3_`);
- `aspcapStar` has the stellar parameters measured from the APOGEE spectrum: effective temperature `teff`, surface gravity `logg` and metallicity `m_h`.

The `WHERE` conditions select:

- stars within the distance of the parameter set (parallax above `min_parallax`): within $200$ parsecs there is little dust to dim and redden the stars;
- parallaxes measured to better than $10\,\%$;
- one row per star: a star observed in two fields or with both APOGEE telescopes has extra rows, flagged with the `DUPLICATE` bit of `extratarg`;
- stars with all three Gaia magnitudes; a missing one is stored as $-9999$.

APOGEE also marks a missing stellar parameter with $-9999$. `NULLIF(..., -9999)` turns it into `NULL`, which pandas reads as `NaN`, so the missing values cannot be mistaken for measurements.

In [ ]:
hr_limits = settings.parameter_set("hr_diagram")
hr_query = f"""
SELECT
    star.gaiaedr3_phot_g_mean_mag
        + 5 * LOG10(star.gaiaedr3_parallax / 100) AS abs_g,
    star.gaiaedr3_phot_bp_mean_mag
        - star.gaiaedr3_phot_rp_mean_mag AS bp_rp,
    NULLIF(aspcap.teff, -9999) AS teff,
    NULLIF(aspcap.logg, -9999) AS logg,
    NULLIF(aspcap.m_h, -9999) AS m_h
FROM apogeeStar AS star
    JOIN aspcapStar AS aspcap ON aspcap.apstar_id = star.apstar_id
WHERE
    star.gaiaedr3_parallax > {hr_limits["min_parallax"]}
    AND star.gaiaedr3_parallax
        > {hr_limits["min_parallax_over_error"]}
        * star.gaiaedr3_parallax_error
    AND (star.extratarg & dbo.fApogeeExtraTarg('DUPLICATE')) = 0
    AND star.gaiaedr3_phot_g_mean_mag > 0
    AND star.gaiaedr3_phot_bp_mean_mag > 0
    AND star.gaiaedr3_phot_rp_mean_mag > 0
"""
print(hr_query)

In [ ]:
%%time
stars = CasJobs.executeQuery(
    sql=hr_query, context=settings.CASJOBS_CONTEXT, format="pandas"
)
print(f"{len(stars):,} stars")
stars.head()

### Missing parameters

The spectroscopic fit does not succeed for every star. The table below counts the stars by what was measured and gives the median position of each group in the HR diagram.

In [ ]:
measured = pd.Series("Teff, log g and [M/H]", index=stars.index)
measured[stars["m_h"].isna()] = "Teff and log g only"
measured[stars["teff"].isna()] = "none"
stars.groupby(measured).agg(
    stars=("abs_g", "size"),
    median_bp_rp=("bp_rp", "median"),
    median_abs_g=("abs_g", "median"),
)

### The diagram

The left panel is the HR diagram as an observer draws it: absolute magnitude against color, with every star. The right panel is the **Kiel diagram**: the temperature and surface gravity measured from the spectra of the same stars, without any distance. Surface gravity is $g = G M / R^{2}$: giants have large radii and low gravity, so they rise toward the top as in the HR diagram. Both panels put hot stars on the left. Color shows the metallicity $[\mathrm{M/H}]$, the abundance of elements heavier than helium relative to the Sun on a log scale; stars without a metallicity are grey. Stars without a temperature are missing from the right panel.

In [ ]:
metal_cmap = plt.get_cmap("RdYlBu_r").with_extremes(bad="0.75")
metal_norm = Normalize(vmin=-0.8, vmax=0.4)
shuffled = stars.sample(frac=1, random_state=0)

with plt.style.context(settings.LIGHT_STYLE):
    fig, (hr_ax, kiel_ax) = plt.subplots(
        1, 2, figsize=(settings.FIGURE_WIDTH, 0.5 * settings.FIGURE_WIDTH)
    )
    for ax, x, y in [(hr_ax, "bp_rp", "abs_g"), (kiel_ax, "teff", "logg")]:
        points = ax.scatter(
            shuffled[x],
            shuffled[y],
            c=shuffled["m_h"],
            cmap=metal_cmap,
            norm=metal_norm,
            s=0.5,
            linewidths=0,
            plotnonfinite=True,
            rasterized=True,
        )
    hr_ax.invert_yaxis()
    hr_ax.set_xlabel(r"color $\mathrm{BP} - \mathrm{RP}$")
    hr_ax.set_ylabel("absolute magnitude $M_G$")
    kiel_ax.set_xscale("log")
    kiel_ax.invert_xaxis()
    kiel_ax.invert_yaxis()
    kiel_ax.xaxis.set_major_locator(LogLocator(subs=(1, 2, 3, 5, 7)))
    kiel_ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    kiel_ax.xaxis.set_minor_formatter(NullFormatter())
    kiel_ax.set_xlabel(r"effective temperature $T_\mathrm{eff}$ (K)")
    kiel_ax.set_ylabel(
        r"surface gravity $\log g$ ($g$ in $\mathrm{cm\,s^{-2}}$)"
    )
    fig.colorbar(
        points,
        ax=[hr_ax, kiel_ax],
        label=r"metallicity $[\mathrm{M/H}]$",
    )
    plt.show()

What to look for:

- **Main sequence.** The diagonal band from hot, luminous stars at the upper left to cool, faint M dwarfs at the lower right. In the Kiel diagram it is the nearly flat band at $\log g \approx 4.5$. Most stars of this sample are on it.
- **Metallicity across the main sequence.** At a given color, metal-rich stars lie higher: around $\mathrm{BP} - \mathrm{RP} = 0.9$, the stars with $[\mathrm{M/H}]$ above $+0.1$ are about half a magnitude brighter than those below $-0.5$. A metal-poor star is hotter and bluer than a metal-rich star of the same mass, so at the same color it is a less massive and fainter star.
- **Subgiants and giants.** Stars that have used up the hydrogen in their cores leave the main sequence: they swell and cool, moving up and to the right in the left panel (subgiants at $M_G \approx 2$–$4$) and to lower gravity in the right panel ($\log g \approx 3.6$). Giants make up less than $1\,\%$ of this sample.
- **Red clump.** The knot of giants at $M_G \approx 0.5$, $\mathrm{BP} - \mathrm{RP} \approx 1.2$ is the red clump: giants burning helium in their cores. The same stars form the concentration at $T_\mathrm{eff} \approx 4800\,\mathrm{K}$ and $\log g \approx 2.5$ in the right panel. One group, found once from Gaia's distances and colors and once from the spectra alone.
- **Grey stars.** Most stars with a temperature but no metallicity are M dwarfs redder than $\mathrm{BP} - \mathrm{RP} = 2$ or hot stars above $7000\,\mathrm{K}$. The stars with no spectroscopic parameters at all (see the table above) are mostly M dwarfs too, with the rest spread along the main sequence.
- **A target list, not a census.** The numbers of stars of each kind reflect what APOGEE chose to observe, not the make-up of the solar neighborhood. Most of the hot stars at the top of the main sequence ($87\,\%$ of those bluer than $\mathrm{BP} - \mathrm{RP} = 0.5$) are calibration stars, observed to measure the absorption lines of the Earth's atmosphere.